# Chatbot RAG Finance + Recherche Web

**Architecture** :
```
Question → Llama3.2 (Ollama) → choisit un outil :
                                ├── rag_finance  (FAISS + PDFs Banque de France / INC)
                                └── web_search   (DuckDuckGo, infos récentes) ← BONUS
```

**Stack** : LangChain 1.x + LangGraph + Ollama (llama3.2:3b) + FAISS + HuggingFace embeddings (`gte-small`) + DuckDuckGo

> ⚠️ **Pourquoi llama3.2:3b et pas llama3 ?** Le tool calling natif (que l'agent utilise pour appeler `rag_finance` ou `web_search`) n'est supporté qu'à partir de llama3.1+. Le `llama3:latest` (8B v1) renvoie une erreur 400 *"does not support tools"*. Llama3.2 est plus petit mais fait le boulot.

## 1. Imports

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

## 2. Charger l'index FAISS

Important : on utilise **le même modèle d'embeddings** que celui qui a généré l'index, sinon les vecteurs sont incompatibles. D'après `Séance 1.ipynb` (cellule 49) c'est `thenlper/gte-small`.

In [ ]:
EMBEDDING_MODEL_NAME = "thenlper/gte-small"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Index FAISS chargé : {vectorstore.index.ntotal} vecteurs")

## 3. Brancher le LLM (llama3.2:3b via Ollama)

In [ ]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0.1,
    base_url="http://localhost:11434",
)

print(llm.invoke("Réponds en une phrase : c'est quoi un livret A ?").content)

## 4. Définir les deux outils

Avec le décorateur `@tool`, la **docstring** sert de description à l'agent : c'est elle qui lui permet de choisir le bon outil. À soigner.

In [ ]:
ddg = DuckDuckGoSearchRun()

@tool
def rag_finance(query: str) -> str:
    """Recherche dans la base de connaissances officielle d'éducation financière
    (Banque de France, INC). À utiliser pour les définitions, concepts financiers
    généraux, produits d'épargne, crédit, budget, arnaques, gestion personnelle."""
    docs = retriever.invoke(query)
    if not docs:
        return "Aucun document pertinent trouvé."
    return "\n\n".join(
        f"[{d.metadata.get('source','?').split('/')[-1]}] {d.page_content[:400]}"
        for d in docs
    )

@tool
def web_search(query: str) -> str:
    """Recherche sur Internet via DuckDuckGo. À utiliser UNIQUEMENT pour des informations
    récentes : taux d'intérêt actuels, cours de bourse, actualité 2025-2026, dernières lois,
    chiffres datés. Ne pas utiliser pour des définitions générales."""
    return ddg.invoke(query)

# Test rapide des outils
print("--- rag_finance ---")
print(rag_finance.invoke("livret A")[:300])
print("\n--- web_search ---")
print(web_search.invoke("taux livret A 2026")[:300])

## 5. Créer l'agent

On utilise `create_react_agent` de LangGraph qui exploite le **tool calling natif** de llama3.2 (bien plus fiable qu'un prompt ReAct textuel).

In [ ]:
tools = [rag_finance, web_search]

SYSTEM = (
    "Tu es un assistant spécialisé en éducation financière. "
    "Tu réponds toujours en français, de manière claire et pédagogique. "
    "Pour les concepts et définitions financières, utilise rag_finance. "
    "Pour les chiffres récents (taux, cours), utilise web_search. "
    "Si tu n'as pas l'information, dis-le clairement plutôt que d'inventer."
)

agent = create_react_agent(llm, tools=tools, prompt=SYSTEM)

## 6. Tests

Petite fonction utilitaire pour afficher le déroulé de l'agent (quel outil il a choisi, etc.).

In [ ]:
def ask(question: str):
    print(f"\n❓ {question}\n" + "─" * 60)
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    for m in result["messages"]:
        kind = type(m).__name__
        if kind == "AIMessage" and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"🔧 Outil appelé : {tc['name']}({tc['args']})")
        elif kind == "ToolMessage":
            print(f"📄 Résultat outil ({len(str(m.content))} chars)")
        elif kind == "AIMessage" and m.content:
            print(f"\n💬 RÉPONSE :\n{m.content}")

### Test 1 — Question "définition" → l'agent doit choisir `rag_finance`

In [ ]:
ask("C'est quoi un Livret A et à qui s'adresse-t-il ?")

### Test 2 — Question "actualité" → l'agent doit choisir `web_search`

In [ ]:
ask("Quel est le taux du Livret A en 2026 ?")

### Test 3 — Question hors finance → l'agent doit refuser ou dire qu'il ne sait pas

In [ ]:
ask("Comment cuisiner une omelette ?")